In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [37]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Write a python/json/regex Description of task",
        "format": "python/json/regex",
        "solution_criteria": "Description of what a correct solution should have"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)
    

In [38]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [35]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task: {test_case['task']}
    * Avoid adding extra info, or description inside the function or solution, only the main description is enough
    * Response only what is asked, avoid adding examples or any other extra information
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    return chat(messages, stop_sequences=["```"])

In [ ]:
def grade_by_model(test_case, output):
    """Grades the output of a test case by asking the model to grade it"""
    prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

What the solution should accomplish:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [26]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [40]:
def run_test_cases(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)
    # print(f"Model Score: {model_score}, Syntax Score: {syntax_score}, Test Case: {test_case['task']}, output: {output}")
    score = (model_score + syntax_score) / 2
    return {
        "output": output,
        "score": score,
        "test_case": test_case,
        "reasoning": model_grade["reasoning"],
    }

In [23]:
from statistics import mean
def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_cases(test_case)
        results.append(result)
    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    return results

In [39]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

Model Score: 7, Syntax Score: 10, Test Case: Write a Python function that takes an AWS S3 bucket name and returns True if it follows AWS naming conventions (lowercase, 3-63 characters, no consecutive hyphens), output: 
import re

def is_valid_s3_bucket_name(bucket_name: str) -> bool:
    if not isinstance(bucket_name, str):
        return False
    
    if len(bucket_name) < 3 or len(bucket_name) > 63:
        return False
    
    if bucket_name != bucket_name.lower():
        return False
    
    if not re.match(r'^[a-z0-9.-]+$', bucket_name):
        return False
    
    if '--' in bucket_name:
        return False
    
    if bucket_name.startswith('-') or bucket_name.endswith('-'):
        return False
    
    if bucket_name.startswith('.') or bucket_name.endswith('.'):
        return False
    
    return True

Model Score: 7, Syntax Score: 10, Test Case: Create a JSON configuration object for an AWS Lambda function that specifies the function name, runtime (Python 3.11), memo

In [25]:
print(json.dumps(results, indent=2))

[
  {
    "output": "```python\nimport re\n\ndef validate_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates an AWS S3 bucket name according to AWS naming rules.\n    \n    Rules:\n    - Length: 3-63 characters\n    - Allowed characters: lowercase letters, numbers, hyphens\n    - Cannot start or end with hyphen\n    \"\"\"\n    if not bucket_name or not isinstance(bucket_name, str):\n        return False\n    \n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return False\n    \n    if bucket_name.startswith('-') or bucket_name.endswith('-'):\n        return False\n    \n    pattern = r'^[a-z0-9-]+$'\n    return bool(re.match(pattern, bucket_name))\n\n\n# Test cases\nif __name__ == \"__main__\":\n    test_cases = [\n        (\"my-bucket\", True),\n        (\"mybucket123\", True),\n        (\"a1b2c3\", True),\n        (\"valid-bucket-name\", True),\n        (\"ab\", False),  # too short\n        (\"a\" * 64, False),  # too long\n        (\"-invalid\", Fal